# RandomForestClassifier → fail-closed decision model (binary **and** multiclass)

**Two problems, one harness.**

1. **Binary** — `breast_cancer` (all 30 features): decide whether a case is safe to call *benign*
   without human review. `p_positive` is P(malignant).
2. **Multiclass** — `wine` (all 13 features): decide whether a sample is the approved class 0;
   `pred`/`conf` and the conformal prediction **set** carry the uncertainty.

Feature names are sanitized (spaces and `/` → `_`) so they are valid DSL lambda parameters — the model
still uses **all** features.

**The shape of every policy is the same:** stated facts + model facts (`pred`, `p_positive`, `conf`,
`certified`, `set_size`) → `release | ask | deny | abstain`. `certified` means the conformal prediction
set is a singleton at α = 0.10; a non-singleton set cannot be released.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
from sklearn.datasets import load_breast_cancer, load_wine
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

from mjolnir_ml import (ASK, DENY, RELEASE, DecisionModel, DecisionPolicyBase,
                        HazardLedger, dsl, report, rule, certify, verify_certificate)

pd.set_option("display.width", 140)
clean = lambda names: tuple(n.replace(" ", "_").replace("/", "_") for n in names)

## Binary — breast_cancer (all 30 features)

Policy (four statements, all four verdicts represented):

- `deny_out_of_spec` — a **stated-fact** hazard (`worst_area > 1500`): a measurement out of spec is refused.
- `deny_confident_malignant` — a confident risky prediction is refused.
- `release_certified_benign` — release only when the conformal set is a **singleton** at α and the risk is low.
- `review_unsettled` — a non-certified prediction escalates (`ask`).

The oracle for the Ω pair is `safe = (label == 0)` (benign).

In [2]:
data = load_breast_cancer()
FEATURES_B = clean(data.feature_names)
Xtr, Xrest, ytr, yrest = train_test_split(data.data, data.target, test_size=0.5,
                                          stratify=data.target, random_state=0)
Xcal, Xte, ycal, yte = train_test_split(Xrest, yrest, test_size=0.5,
                                        stratify=yrest, random_state=0)
rows = lambda A: [dict(zip(FEATURES_B, map(float, x))) for x in A]
tr_b, cal_b, te_b = rows(Xtr), rows(Xcal), rows(Xte)

forest_b = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(Xtr, ytr)
print("features (all):", FEATURES_B)

@dsl
class MammographyPolicy(DecisionPolicyBase):
    assert rule("deny_out_of_spec",
                when=lambda worst_area: worst_area > 1500.0, verdict=DENY)
    assert rule("deny_confident_malignant",
                when=lambda p_positive, conf: p_positive >= 0.80 and conf >= 0.80, verdict=DENY)
    assert rule("release_certified_benign",
                when=lambda certified, p_positive: certified and p_positive < 0.50, verdict=RELEASE)
    assert rule("review_unsettled",
                when=lambda certified: not certified, verdict=ASK)

dm_b = (DecisionModel(forest_b, MammographyPolicy, features=FEATURES_B)
        .fit_regime(tr_b).calibrate(cal_b, ycal, alpha=0.10))

ledger_b = (HazardLedger()
            .add("out_of_spec_area", lambda r: r["worst_area"] > 1500.0,
                 "spec sheet §4.1"))
print(report(dm_b, te_b, title="Breast cancer — release benign without review",
             safe=[int(v) == 0 for v in yte], decidable=[True] * len(yte),
             ledger=ledger_b, key=b"demo-audit-key", sample=8))

features (all): ('mean_radius', 'mean_texture', 'mean_perimeter', 'mean_area', 'mean_smoothness', 'mean_compactness', 'mean_concavity', 'mean_concave_points', 'mean_symmetry', 'mean_fractal_dimension', 'radius_error', 'texture_error', 'perimeter_error', 'area_error', 'smoothness_error', 'compactness_error', 'concavity_error', 'concave_points_error', 'symmetry_error', 'fractal_dimension_error', 'worst_radius', 'worst_texture', 'worst_perimeter', 'worst_area', 'worst_smoothness', 'worst_compactness', 'worst_concavity', 'worst_concave_points', 'worst_symmetry', 'worst_fractal_dimension')


 Breast cancer — release benign without review
 mjolnir.ml.report/1 · 2026-10-22T16:48:21Z

CONTRACT (the problem)
----------------------
  policy        MammographyPolicy  (sha256:8827e9656cd34828a358dc20affc760ba36b20b79b6add45d2068f7348727313)
  model         RandomForestClassifier  (sha256:03ae4b5b6409c5b0…)
  calibration   alpha=0.10  (sha256:b615e54de3c4057e…)
  regime        required   features=30
  features      mean_radius, mean_texture, mean_perimeter, mean_area, mean_smoothness, mean_compactness, mean_concavity, mean_concave_points, mean_symmetry, mean_fractal_dimension, radius_error, texture_error, perimeter_error, area_error, smoothness_error, compactness_error, concavity_error, concave_points_error, symmetry_error, fractal_dimension_error, worst_radius, worst_texture, worst_perimeter, worst_area, worst_smoothness, worst_compactness, worst_concavity, worst_concave_points, worst_symmetry, worst_fractal_dimension

POLICY (every statement, compiled)
--------------------------

## Multiclass — wine (all 13 features)

For a multiclass classifier there is no `p_positive`; `conf` is the max probability and the **conformal set** is the
interesting object. Rules can use `pred` (the point label) and `certified` (set singleton). We add a
stated-fact rule on `alcohol` to show feature facts next to model facts.

In [3]:
data = load_wine()
FEATURES_W = clean(data.feature_names)
Xtr, Xrest, ytr, yrest = train_test_split(data.data, data.target, test_size=0.5,
                                          stratify=data.target, random_state=0)
Xcal, Xte, ycal, yte = train_test_split(Xrest, yrest, test_size=0.5,
                                        stratify=yrest, random_state=0)
rows = lambda A: [dict(zip(FEATURES_W, map(float, x))) for x in A]
tr_w, cal_w, te_w = rows(Xtr), rows(Xcal), rows(Xte)

forest_w = RandomForestClassifier(n_estimators=300, random_state=0, n_jobs=-1).fit(Xtr, ytr)
print("features (all):", FEATURES_W)

@dsl
class WinePolicy(DecisionPolicyBase):
    assert rule("deny_high_alcohol",
                when=lambda alcohol: alcohol > 14.5, verdict=DENY)
    assert rule("deny_off_profile",
                when=lambda pred, conf: pred == 2 and conf >= 0.80, verdict=DENY)
    assert rule("release_approved",
                when=lambda certified, pred: certified and pred == 0, verdict=RELEASE)
    assert rule("review_unsettled",
                when=lambda certified: not certified, verdict=ASK)

dm_w = (DecisionModel(forest_w, WinePolicy, features=FEATURES_W)
        .fit_regime(tr_w).calibrate(cal_w, ycal, alpha=0.10))

ledger_w = (HazardLedger()
            .add("high_alcohol", lambda r: r["alcohol"] > 13.5,
                 "product spec §1 (alcohol by volume)"))
print(report(dm_w, te_w, title="Wine — approve class 0, refuse the rest",
             safe=[int(v) == 0 for v in yte], decidable=[True] * len(yte),
             ledger=ledger_w, key=b"demo-audit-key", sample=8))

features (all): ('alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280_od315_of_diluted_wines', 'proline')


 Wine — approve class 0, refuse the rest
 mjolnir.ml.report/1 · 2026-10-22T16:48:22Z

CONTRACT (the problem)
----------------------
  policy        WinePolicy  (sha256:1260a379e70d68662f5f977f13b3f9b4247ce8e7c451d35e49490a57297d725a)
  model         RandomForestClassifier  (sha256:9cedeb416c1b59be…)
  calibration   alpha=0.10  (sha256:4670ddc4cadaa0c2…)
  regime        required   features=13
  features      alcohol, malic_acid, ash, alcalinity_of_ash, magnesium, total_phenols, flavanoids, nonflavanoid_phenols, proanthocyanins, color_intensity, hue, od280_od315_of_diluted_wines, proline

POLICY (every statement, compiled)
----------------------------------
  code               kind     requires         predicate                  verdict
  -----------------  -------  ---------------  -------------------------  -------
  deny_high_alcohol  rule     alcohol          alcohol > 14.5             deny   
  deny_off_profile   rule     pred, conf       pred == 2 and conf >= 0.8  deny   
  releas

## What to read in the report

- **DECISIONS** — how often the harness committed, escalated or refused; abstention causes are named
  (`out_of_regime`, `missing fact`, `uncovered`, premise codes).
- **OMEGA PAIR** — precision/recall of *claims* against the pinned oracle, with Wilson intervals, beside
  the degenerate ALWAYS/NEVER controls (NEVER has vacuous precision `⊥`, not 1).
- **HAZARDS** — completeness against the external ledger: a release on a hazard row is a false
  certificate. `observable recall` vs `hidden-only recall` separates declared coverage from luck.
- **CERTIFICATES** — signed and replayable counts; the auditor needs the policy artifact and the key,
  not the forest.

In [4]:
# The certificate is the artifact: one row, checked without the estimator.
cert = certify(dm_w.dispose(te_w[0]), key=b"demo-audit-key")
print("certificate id :", cert["certificate_id"][:34], "…")
print("verdict        :", cert["verdict"], "| basis:", cert["basis"])
print("facts          :", list(cert["facts"])[:5], "…")
print("verified       :", verify_certificate(cert, key=b"demo-audit-key",
                                            policy=WinePolicy))
print("replayable     :", cert["replayable"])

certificate id : sha256:2cb010fbfa5a3826d8a9c9c809d …
verdict        : release | basis: ['release_approved']
facts          : ['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium'] …
verified       : True
replayable     : True


## Takeaways

- The same harness handles binary and multiclass classifiers; only the policy changes.
- `certified` is a finite-sample guarantee (a conformal singleton), not a softmax threshold — that is
  what makes a release defensible.
- Stated-fact rules (`worst_area`, `alcohol`) sit beside model facts (`p_positive`, `conf`, `certified`) in
  one declared, digested table; `report()` shows both.
- Swap RandomForest for LogisticRegression / HistGradientBoosting: the policy, κ and certificates are
  unchanged; only `model_digest` moves.